# 05 - Training and evaluation

**Goals:** connect batches, losses, gradients, optimizer steps, and validation learning curves.

Combine the batches from [03](03_data_and_batches.ipynb), models from [04](04_models_and_losses.ipynb), and gradients from [02](02_autograd.ipynb). Run cells in order in a fresh kernel; each notebook supplies its own setup.

## Prepare the data

Recreate the training-fitted tensors from notebook 03. Test rows stay reserved for the final assessment in notebook 08.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(486)
torch.set_num_threads(1)  # Keep these small CPU examples quick.


In [ ]:
root = Path.cwd()
if not (root / "data" / "observations.csv").exists():
    root = root.parent
data_path = root / "data" / "observations.csv"
assert data_path.exists(), "Start Jupyter in the repository root or notebooks directory."

df = pd.read_csv(data_path)

features = ["feature_1", "feature_2", "feature_3"]
train = df[df["split"] == "train"]
validation = df[df["split"] == "validation"]
test = df[df["split"] == "test"]


In [ ]:
scaler = StandardScaler().fit(train[features])  # Fit only on training rows.

def tensors(rows):
    x = torch.tensor(scaler.transform(rows[features]), dtype=torch.float32)
    y = torch.tensor(rows["label"].to_numpy(), dtype=torch.long)
    return x, y

x_train, y_train = tensors(train)
x_val, y_val = tensors(validation)


## Connect loaders, model, and optimizer

Use shuffled batches to train and fixed-order loaders to measure performance. A separate training evaluation loader measures every row using the current model.

In [ ]:
train_data = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
train_eval_loader = DataLoader(train_data, batch_size=64, shuffle=False)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=64, shuffle=False)


Use a hidden width of 16 for this baseline. Adam manages the parameter updates that we performed manually in notebook 02; constructing it does not update any weights.

In [ ]:
model = nn.Sequential(nn.Linear(len(features), 16), nn.ReLU(), nn.Linear(16, 2))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


## Inspect one training step

First compute the logits and loss for one batch. A step will update parameters once; an epoch will visit every batch.

In [ ]:
model.train()
x_batch, y_batch = next(iter(train_loader))
optimizer.zero_grad()
logits = model(x_batch)
loss = loss_fn(logits, y_batch)
print("batch and logits shapes:", x_batch.shape, logits.shape)
print("loss before update:", loss.item())


`backward()` computes gradients; `step()` applies them. Inspect a weight gradient before the update.

In [ ]:
loss.backward()
print("first weight gradient shape:", model[0].weight.grad.shape)
optimizer.step()


## Train one epoch

Reset both model and optimizer to start the full experiment without the demonstration update. The helper keeps the same sequence for every batch. Edit this reset cell when changing settings for the full run.

In [ ]:
torch.manual_seed(486)
model = nn.Sequential(nn.Linear(len(features), 16), nn.ReLU(), nn.Linear(16, 2))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


In [ ]:
def train_one_epoch(model, loader):
    model.train()
    for x_batch, y_batch in loader:
        optimizer.zero_grad()  # Each batch needs fresh gradients.
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()


## Measure performance

`eval()` changes the behavior of layers such as dropout; `no_grad()` stops recording a gradient graph. Neither performs an update. Multiply each mean batch loss by its size so the smaller final batch has the correct contribution.

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            logits = model(x_batch)
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()
            count += len(y_batch)
    return total_loss / count, correct / count


## Repeat epochs and track validation

Train for 20 passes through the training loader. The helper calls `train()` again after evaluation. Both losses below are measured after each epoch with the same current weights in evaluation mode.

In [ ]:
history = []
for epoch in range(1, 21):
    train_one_epoch(model, train_loader)
    train_loss, train_accuracy = evaluate(model, train_eval_loader)
    val_loss, val_accuracy = evaluate(model, val_loader)
    history.append((epoch, train_loss, val_loss, train_accuracy, val_accuracy))

print(f"Epoch {epoch}: train loss={train_loss:.3f}, validation loss={val_loss:.3f}, "
      f"validation accuracy={val_accuracy:.3f}")


Store the metrics so that the trajectory, rather than just the final value, can guide the next experiment.

In [ ]:
history_frame = pd.DataFrame(
    history, columns=["epoch", "train_loss", "val_loss", "train_accuracy", "val_accuracy"]
)
history_frame.tail()


Compare the curves: falling training loss with rising validation loss can indicate overfitting. Similar high losses may suggest underfitting or an optimization problem. Neither pattern alone determines the cause.

In [ ]:
history_frame.plot(x="epoch", y=["train_loss", "val_loss"],
                   ylabel="Cross-entropy loss", title="Learning curves")
plt.show()


Use validation to choose settings. Test evaluation is deferred to notebook 08, after the final approach has been selected.

## Try it

1. In the full-run setup, replace Adam with `torch.optim.SGD` and try learning rates. Compare the curves.

2. Print gradient norms before `optimizer.step()`. What happens if `zero_grad()` is omitted?

3. Change batch size and count updates per epoch. Why is an epoch different from a step?

4. Add a confusion matrix; explain what accuracy alone can hide.

**Next:** [Regularization](06_regularization.ipynb) adds controls for overfitting and selects an epoch using validation loss.